# Solutions — Interfaces → Duck Typing, Protocols, ABCs & Dunder Methods

Exercise notebook: `04_java_to_python/04_interfaces_vs_duck_typing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · If it has .size(), it's a source

**Working solution**

In [ ]:
def total_size(sources):
    total = 0
    for source in sources:
        size = getattr(source, "size", None)
        if callable(size):
            total += size()
    return total

**Explanation**

Duck typing: behaviour, not type, decides. No `implements Sized`. `getattr` with a default is the safe 'does it have it?' probe.

**Complexity:** O(n).

**Common mistakes**

- `isinstance(source, SomeBase)` checks that reject perfectly good objects.

## Exercise 2 — 🟡 Medium · A Sink protocol with interchangeable implementations

**Working solution**

In [ ]:
import csv
from pathlib import Path
from typing import Protocol

class Sink(Protocol):
    def write(self, records: list[dict]) -> int: ...

class ListSink:
    def __init__(self):
        self.rows = []

    def write(self, records):
        self.rows.extend(records)
        return len(records)

class CsvSink:
    def __init__(self, path, fieldnames):
        self.path = Path(path)
        self.fieldnames = list(fieldnames)
        self._header_written = self.path.exists() and self.path.stat().st_size > 0

    def write(self, records):
        with self.path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            if not self._header_written:
                writer.writeheader()
                self._header_written = True
            writer.writerows(records)
        return len(records)

def run_export(records, sink: Sink, batch_size=2) -> int:
    records = list(records)
    return sum(sink.write(records[i:i + batch_size]) for i in range(0, len(records), batch_size))

**Explanation**

`Protocol` is a *structural* interface: mypy/pyright accept any object with a matching `write`, no inheritance needed. At runtime it's
pure duck typing. This is how you make code testable: pass a `ListSink` in tests, a `CsvSink`/`S3Sink` in production.

**Complexity:** O(n).

**Common mistakes**

- Writing the CSV header on every call.
- Making `run_export` check `isinstance(sink, CsvSink)`.

## Exercise 3 — 🟡 Medium · Abstract base classes for validators

**Working solution**

In [ ]:
from abc import ABC, abstractmethod

class Validator(ABC):
    @abstractmethod
    def validate(self, record: dict) -> list[str]:
        ...

    def __call__(self, record: dict) -> list[str]:
        return self.validate(record)

class RequiredFields(Validator):
    def __init__(self, *fields):
        self.fields = fields

    def validate(self, record):
        return [f"{f} is required" for f in self.fields if record.get(f) is None or str(record.get(f)).strip() == ""]

class RangeCheck(Validator):
    def __init__(self, field, lo, hi):
        self.field, self.lo, self.hi = field, lo, hi

    def validate(self, record):
        if record.get(self.field) is None:
            return []
        try:
            value = float(record[self.field])
        except (TypeError, ValueError):
            return [f"{self.field} is not numeric"]
        return [] if self.lo <= value <= self.hi else [f"{self.field} out of range [{self.lo}, {self.hi}]"]

def validate_all(record, validators):
    return [error for v in validators for error in v(record)]

**Explanation**

Use an ABC when you *own* a framework and want subclasses to fail fast if they forget a method — `TypeError` at instantiation, not at
3 a.m. in production. Note `validate_all` calls `v(record)`, so plain functions returning lists also work as validators: ABC for the
framework author, duck typing for the user.

**Complexity:** O(validators × fields).

**Common mistakes**

- Using ABCs everywhere 'like Java interfaces' — most of the time a Protocol or nothing is enough.

## Exercise 4 — 🟠 Hard · Value objects through dunder methods

**Working solution**

In [ ]:
from decimal import Decimal
from functools import total_ordering

@total_ordering
class Money:
    __slots__ = ("_amount", "_currency")

    def __init__(self, amount, currency):
        if isinstance(amount, float):
            raise TypeError("use str or Decimal for money, not float")
        object.__setattr__(self, "_amount", Decimal(amount))
        object.__setattr__(self, "_currency", currency.upper())

    def __setattr__(self, name, value):
        raise AttributeError("Money is immutable")

    @property
    def amount(self):
        return self._amount

    @property
    def currency(self):
        return self._currency

    def _check(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if other.currency != self.currency:
            raise ValueError(f"currency mismatch: {self.currency} vs {other.currency}")
        return other

    def __add__(self, other):
        other = self._check(other)
        if other is NotImplemented:
            return NotImplemented
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        if other == 0:
            return self
        return self.__add__(other)

    def __sub__(self, other):
        other = self._check(other)
        if other is NotImplemented:
            return NotImplemented
        return Money(self.amount - other.amount, self.currency)

    def __mul__(self, factor):
        if isinstance(factor, (int, Decimal)) and not isinstance(factor, bool):
            return Money(self.amount * factor, self.currency)
        return NotImplemented

    __rmul__ = __mul__

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __lt__(self, other):
        other = self._check(other)
        if other is NotImplemented:
            return NotImplemented
        return self.amount < other.amount

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __repr__(self):
        return f"Money('{self.amount}', '{self.currency}')"

    def __str__(self):
        return f"{self.currency} {self.amount.quantize(Decimal('0.01'))}"

**Explanation**

Operators in Python are just methods: `a + b` calls `a.__add__(b)`, then `b.__radd__(a)` if the first returns `NotImplemented`. `sum()` starts
from `0`, so `__radd__` must accept `0`. Returning `NotImplemented` (not raising) lets Python try the other operand. `total_ordering` derives
`<=`, `>`, `>=` from `__eq__` + `__lt__`. `Decimal('10.5') == Decimal('10.50')`, so equal amounts hash equally.

**Complexity:** O(1).

**Common mistakes**

- Raising `TypeError` instead of returning `NotImplemented`.
- Defining `__eq__` without `__hash__` → instances become unhashable.

## Challenge — 🔴 Very Hard · Strategy pattern → plain callables

**Working solution**

In [ ]:
from bisect import bisect_right

def percent_off(pct, min_total=0):
    def strategy(order):
        return order["total"] * pct / 100 if order["total"] >= min_total else 0
    strategy.__name__ = f"percent_off_{pct:g}"
    return strategy

class TieredDiscount:
    def __init__(self, tiers):
        ordered = sorted(tiers)
        self.thresholds = [t for t, _ in ordered]
        self.discounts = [d for _, d in ordered]

    def __call__(self, order):
        pos = bisect_right(self.thresholds, order["total"]) - 1
        return self.discounts[pos] if pos >= 0 else 0

def best_price(order, strategies):
    total = order["total"]
    best_discount, best_name = 0, "none"
    for strategy in strategies:
        discount = min(strategy(order), total)
        if discount > best_discount:
            best_discount = discount
            best_name = getattr(strategy, "__name__", type(strategy).__name__)
    return round(total - best_discount, 2), best_name

**Explanation**

Functions are objects, closures capture configuration (`pct`, `min_total`), and any class with `__call__` is callable too. Three Java
classes + an interface become one factory function and one small class. Mixing both in the same list is fine — that's duck typing.

**Complexity:** O(strategies × log tiers).

**Common mistakes**

- Creating an abstract `Strategy` base class nobody needs.
- Letting a flat discount push the price negative.